# Modul 18: CNNs und Transfer Learning mit PyTorch | Lösungen

## Überblick

Sie erstellen Bild-Datasets und DataLoader, implementieren ein LeNet-ähnliches CNN und analysieren Fehlerbilder. Danach testen Sie Augmentation, Batch Normalization, Dropout, Lernratensteuerung und Early Stopping sowie MobileNetV3 Small als eingefrorenen Transfer-Learning-Extraktor.

**Zugehörige Vorlesungen**

- **LeNet mit PyTorch**
- **Transfer mit PyTorch**

## Lernziele

Nach der Bearbeitung können Sie:

- kleine Bilddaten als kanalzuerst angeordnete PyTorch-Tensoren mit getrennten Trainings- und Evaluationstransformationen vorbereiten.
- ein LeNet-ähnliches nn.Module trainieren und mit Konfusionsmatrix sowie Fehlbildern bewerten.
- Regularisierung und Transfer Learning kontrolliert einsetzen und Modelle nach Leistung, Laufzeit und Größe vergleichen.

## Geprüfte Fähigkeiten

- Bild-Dataset, DataLoader, Augmentation und Conv2d-Formen
- PyTorch-CNN, BatchNorm, Dropout, Scheduler, Early Stopping und Fehleranalyse
- torchvision MobileNetV3 Small, Einfrieren, Kopftraining, Inferenzzeit und Offline-Fallback

## Hinweise zur Bearbeitung

Dieses Lösungsnotebook enthält dieselben Aufgaben wie das Übungsnotebook sowie vollständige, ausführlich kommentierte Musterlösungen. Bearbeiten Sie nach Möglichkeit zuerst das Übungsnotebook und nutzen Sie dieses Dokument anschließend zur Kontrolle und Vertiefung.

- **Erwarteter Schwierigkeitsgrad:** anspruchsvoll
- Verwenden Sie sprechende Variablennamen und prüfen Sie wichtige Zwischenformen und Wertebereiche.
- Verändern Sie die vorgegebenen Zufalls-Startwerte nur, wenn eine Aufgabe dies ausdrücklich verlangt.
- Interpretieren Sie Ergebnisse fachlich. Eine einzelne Kennzahl ist selten eine vollständige Begründung.
- Alle Aufgaben sind für die kostenlose Google-Colab-Umgebung ausgelegt. Die Datensätze und Modelle sind bewusst klein gehalten. Eine GPU ist nicht erforderlich, kann aber bei einzelnen Deep-Learning-Aufgaben die Laufzeit verkürzen.

## Einrichtung und gemeinsame Datenbasis

Die Setup-Zelle lädt den kleinen Digits-Datensatz, teilt ihn reproduzierbar und wandelt die Pixel in Float32-Tensoren im Bereich 0 bis 1 um. Alle Standardaufgaben laufen auf der CPU. Der optionale Download vortrainierter MobileNet-Gewichte besitzt einen dokumentierten Offline-Fallback.

In [ ]:
import copy
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision
from torchvision import models

from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, ConfusionMatrixDisplay
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
geraet = torch.device("cpu")

ziffern = load_digits()
bilder_gesamt = (ziffern.images.astype("float32") / 16.0)
labels_gesamt = ziffern.target.astype("int64")

bilder_train_np, bilder_test_np, labels_train_np, labels_test_np = train_test_split(
    bilder_gesamt,
    labels_gesamt,
    test_size=0.20,
    stratify=labels_gesamt,
    random_state=RANDOM_SEED,
)
bilder_train_np, bilder_val_np, labels_train_np, labels_val_np = train_test_split(
    bilder_train_np,
    labels_train_np,
    test_size=0.20,
    stratify=labels_train_np,
    random_state=RANDOM_SEED,
)

print("PyTorch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("Train, Validierung, Test:", bilder_train_np.shape, bilder_val_np.shape, bilder_test_np.shape)

### Aufgabe 1: Trainings- und Evaluationstransformationen trennen

Definieren Sie eine Dataset-Klasse für die 8-mal-8-Bilder. Jedes Bild soll als Tensor der Form `(1, 8, 8)` zurückgegeben werden. Im Trainingsmodus sollen mit jeweils kleiner Wahrscheinlichkeit eine Verschiebung um höchstens ein Pixel und schwaches Gaußrauschen angewendet werden. Werte müssen anschließend auf 0 bis 1 begrenzt werden.

Validierung und Test dürfen keine zufällige Augmentation erhalten. Erstellen Sie Loader mit Batchgröße 32 und visualisieren Sie ein Originalbild sowie drei augmentierte Varianten desselben Trainingsbildes.

In [ ]:
class DigitsDataset(Dataset):
    def __init__(self, bilder, labels, augment=False, seed=42):
        pass

    def __len__(self):
        pass

    def __getitem__(self, index):
        pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

class DigitsDataset(Dataset):
    def __init__(self, bilder, labels, augment=False, seed=42):
        self.bilder = torch.tensor(bilder, dtype=torch.float32).unsqueeze(1)
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.augment = augment
        self.generator = torch.Generator().manual_seed(seed)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        bild = self.bilder[index].clone()
        label = self.labels[index]

        if self.augment:
            # Kleine Verschiebungen sind für handgeschriebene Ziffern plausibel.
            if torch.rand((), generator=self.generator) < 0.70:
                verschiebung_y = int(torch.randint(-1, 2, (), generator=self.generator))
                verschiebung_x = int(torch.randint(-1, 2, (), generator=self.generator))
                bild = torch.roll(bild, shifts=(verschiebung_y, verschiebung_x), dims=(1, 2))
            if torch.rand((), generator=self.generator) < 0.50:
                rauschen = torch.randn(bild.shape, generator=self.generator) * 0.04
                bild = bild + rauschen
            bild = torch.clamp(bild, 0.0, 1.0)

        return bild, label

train_dataset_aug = DigitsDataset(bilder_train_np, labels_train_np, augment=True, seed=RANDOM_SEED)
val_dataset_bild = DigitsDataset(bilder_val_np, labels_val_np, augment=False)
test_dataset_bild = DigitsDataset(bilder_test_np, labels_test_np, augment=False)

loader_generator = torch.Generator().manual_seed(RANDOM_SEED)
train_loader_bild = DataLoader(
    train_dataset_aug,
    batch_size=32,
    shuffle=True,
    generator=loader_generator,
    num_workers=0,
)
val_loader_bild = DataLoader(val_dataset_bild, batch_size=64, shuffle=False, num_workers=0)
test_loader_bild = DataLoader(test_dataset_bild, batch_size=64, shuffle=False, num_workers=0)

batch_bilder, batch_labels = next(iter(train_loader_bild))
print("Batchformen:", batch_bilder.shape, batch_labels.shape)
assert batch_bilder.shape[1:] == (1, 8, 8)

plt.figure()
plt.imshow(bilder_train_np[0], cmap="gray", vmin=0, vmax=1)
plt.title("Original")
plt.axis("off")
plt.show()
for wiederholung in range(3):
    augmentiertes_bild, _ = train_dataset_aug[0]
    plt.figure()
    plt.imshow(augmentiertes_bild.squeeze(0), cmap="gray", vmin=0, vmax=1)
    plt.title(f"Augmentierte Variante {wiederholung + 1}")
    plt.axis("off")
    plt.show()

> **Musterantwort und Interpretation**
>
> Validierung und Test sollen eine feste, reproduzierbare Zielverteilung repräsentieren. Zufällige Veränderungen würden die Bewertungsmenge bei jedem Lauf ändern und Modellvergleiche erschweren. Augmentation ist eine Trainingsregularisierung. Ihre Stärke und Plausibilität müssen visuell sowie fachlich geprüft werden.

### Aufgabe 2: Conv2d-Formen verfolgen und LeNet definieren

Wenden Sie eine `nn.Conv2d(1, 16, kernel_size=3, padding=1)` und danach `nn.MaxPool2d(2)` auf einen Batch an. Geben Sie die Formen aus.

Definieren Sie anschließend eine Klasse `KleinesLeNet` mit zwei Conv-Blöcken, ReLU, Pooling, Flatten, einer Dense-Schicht und zehn Logits. Das Modell soll optional BatchNorm und Dropout verwenden. Prüfen Sie die Ausgabeform und ermitteln Sie die Zahl trainierbarer Parameter.

In [ ]:
class KleinesLeNet(nn.Module):
    def __init__(self, use_regularization=False):
        super().__init__()
        pass

    def forward(self, x):
        pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

conv_demo = nn.Conv2d(1, 16, kernel_size=3, padding=1)
pool_demo = nn.MaxPool2d(kernel_size=2, stride=2)
conv_ausgabe = conv_demo(batch_bilder)
pool_ausgabe = pool_demo(torch.relu(conv_ausgabe))
print("Eingabe:", batch_bilder.shape)
print("Nach Conv2d:", conv_ausgabe.shape)
print("Nach Pooling:", pool_ausgabe.shape)


class KleinesLeNet(nn.Module):
    def __init__(self, use_regularization=False):
        super().__init__()
        self.use_regularization = use_regularization
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16) if use_regularization else nn.Identity()
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32) if use_regularization else nn.Identity()
        self.pool = nn.MaxPool2d(2)
        self.flatten = nn.Flatten()
        # 8 -> Pool -> 4 -> Pool -> 2, also 32*2*2 Eingaben.
        self.fc1 = nn.Linear(32 * 2 * 2, 64)
        self.dropout = nn.Dropout(0.25) if use_regularization else nn.Identity()
        self.fc2 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool(torch.relu(self.bn2(self.conv2(x))))
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)

torch.manual_seed(RANDOM_SEED)
lenet_torch = KleinesLeNet(use_regularization=False).to(geraet)
logits_demo = lenet_torch(batch_bilder.to(geraet))
parameterzahl = sum(p.numel() for p in lenet_torch.parameters() if p.requires_grad)
print(lenet_torch)
print("Logitform:", logits_demo.shape)
print("Trainierbare Parameter:", parameterzahl)
assert logits_demo.shape == (len(batch_bilder), 10)

> **Musterantwort und Interpretation**
>
> Linear-Schichten benötigen eine feste Zahl von Eingabemerkmalen. Diese ergibt sich aus Kanälen, Höhe und Breite nach allen Faltungen und Pooling-Schritten. Eine falsche Annahme fällt erst beim Forward-Pass als Formfehler auf. Die explizite Herleitung macht Architekturänderungen nachvollziehbar.

### Aufgabe 3: LeNet auf der CPU trainieren und Fehler analysieren

Schreiben Sie kompakte Trainings- und Auswertungsfunktionen für die Mehrklassenklassifikation mit `CrossEntropyLoss`. Trainieren Sie das unregularisierte LeNet mit Adam für höchstens zwölf Epochen und speichern Sie den besten Validierungszustand.

Berichten Sie Test-Accuracy, Balanced Accuracy und Macro-F1. Erstellen Sie eine Konfusionsmatrix und visualisieren Sie bis zu sechs falsch klassifizierte Testbilder mit ihren Softmax-Sicherheiten.

In [ ]:
def trainiere_cnn_epoche(modell, loader, optimizer, loss_fn, device):
    pass

def bewerte_cnn(modell, loader, loss_fn, device):
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def trainiere_cnn_epoche(modell, loader, optimizer, loss_fn, device):
    modell.train()
    verlust_summe = 0.0
    for bilder_batch, labels_batch in loader:
        bilder_batch = bilder_batch.to(device)
        labels_batch = labels_batch.to(device)
        optimizer.zero_grad()
        logits = modell(bilder_batch)
        verlust = loss_fn(logits, labels_batch)
        verlust.backward()
        optimizer.step()
        verlust_summe += verlust.item() * len(labels_batch)
    return verlust_summe / len(loader.dataset)


def bewerte_cnn(modell, loader, loss_fn, device):
    modell.eval()
    verlust_summe = 0.0
    alle_labels = []
    alle_probs = []
    with torch.no_grad():
        for bilder_batch, labels_batch in loader:
            bilder_batch = bilder_batch.to(device)
            labels_batch = labels_batch.to(device)
            logits = modell(bilder_batch)
            verlust = loss_fn(logits, labels_batch)
            verlust_summe += verlust.item() * len(labels_batch)
            alle_labels.append(labels_batch.cpu().numpy())
            alle_probs.append(torch.softmax(logits, dim=1).cpu().numpy())
    labels_np = np.concatenate(alle_labels)
    probs_np = np.vstack(alle_probs)
    pred_np = probs_np.argmax(axis=1)
    return {
        "loss": verlust_summe / len(loader.dataset),
        "accuracy": accuracy_score(labels_np, pred_np),
        "balanced_accuracy": balanced_accuracy_score(labels_np, pred_np),
        "macro_f1": f1_score(labels_np, pred_np, average="macro"),
        "labels": labels_np,
        "probabilities": probs_np,
        "predictions": pred_np,
    }

torch.manual_seed(RANDOM_SEED)
lenet_torch = KleinesLeNet(use_regularization=False).to(geraet)
loss_cnn = nn.CrossEntropyLoss()
optimizer_cnn = torch.optim.Adam(lenet_torch.parameters(), lr=0.001)

historie_lenet_torch = []
bester_val = np.inf
bester_zustand_lenet = None
for epoche in range(12):
    train_loss = trainiere_cnn_epoche(lenet_torch, train_loader_bild, optimizer_cnn, loss_cnn, geraet)
    val_ergebnis = bewerte_cnn(lenet_torch, val_loader_bild, loss_cnn, geraet)
    historie_lenet_torch.append(
        {"Epoche": epoche + 1, "train_loss": train_loss, "val_loss": val_ergebnis["loss"]}
    )
    if val_ergebnis["loss"] < bester_val:
        bester_val = val_ergebnis["loss"]
        bester_zustand_lenet = copy.deepcopy(lenet_torch.state_dict())

lenet_torch.load_state_dict(bester_zustand_lenet)
test_lenet_torch = bewerte_cnn(lenet_torch, test_loader_bild, loss_cnn, geraet)
print(
    "Test: Accuracy={accuracy:.3f}, Balanced Accuracy={balanced_accuracy:.3f}, Macro-F1={macro_f1:.3f}".format(
        **test_lenet_torch
    )
)

historie_lenet_torch_df = pd.DataFrame(historie_lenet_torch)
plt.plot(historie_lenet_torch_df["Epoche"], historie_lenet_torch_df["train_loss"], label="Training")
plt.plot(historie_lenet_torch_df["Epoche"], historie_lenet_torch_df["val_loss"], label="Validierung")
plt.xlabel("Epoche")
plt.ylabel("CrossEntropyLoss")
plt.title("PyTorch-LeNet-Verlauf")
plt.legend()
plt.show()

ConfusionMatrixDisplay.from_predictions(
    test_lenet_torch["labels"],
    test_lenet_torch["predictions"],
)
plt.title("PyTorch-LeNet-Konfusionsmatrix")
plt.show()

fehler = np.flatnonzero(test_lenet_torch["labels"] != test_lenet_torch["predictions"])[:6]
for index in fehler:
    plt.figure()
    plt.imshow(bilder_test_np[index], cmap="gray", vmin=0, vmax=1)
    plt.title(
        f"Wahr {test_lenet_torch['labels'][index]}, "
        f"vorhergesagt {test_lenet_torch['predictions'][index]}, "
        f"Sicherheit {test_lenet_torch['probabilities'][index].max():.2f}"
    )
    plt.axis("off")
    plt.show()

> **Musterantwort und Interpretation**
>
> CrossEntropyLoss erwartet rohe Logits und kombiniert intern LogSoftmax mit negativer Log-Likelihood. Diese Kombination ist numerisch stabiler. Softmax wird erst für interpretierbare Wahrscheinlichkeiten in der Auswertung berechnet.

### Aufgabe 4: Regularisierung, Scheduler und Early Stopping kontrolliert einsetzen

Trainieren Sie die regularisierte LeNet-Variante mit BatchNorm und Dropout. Nutzen Sie Adam, `ReduceLROnPlateau` auf dem Validierungsverlust und Early Stopping mit Geduld 4. Speichern Sie pro Epoche die Lernrate.

Vergleichen Sie unregularisiertes und regularisiertes Modell hinsichtlich bester Validierungsverlust, Test-Macro-F1 und Anzahl trainierbarer Parameter. Erklären Sie, weshalb BatchNorm im Evaluationsmodus wichtig ist.

In [ ]:
# Verwenden Sie KleinesLeNet(use_regularization=True).

# ============================================================
# MUSTERLÖSUNG
# ============================================================

torch.manual_seed(RANDOM_SEED)
reg_lenet = KleinesLeNet(use_regularization=True).to(geraet)
optimizer_reg = torch.optim.Adam(reg_lenet.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_reg,
    mode="min",
    factor=0.5,
    patience=2,
)

historie_reg = []
bester_val_reg = np.inf
bester_zustand_reg = None
ohne_verbesserung = 0
for epoche in range(20):
    train_loss = trainiere_cnn_epoche(reg_lenet, train_loader_bild, optimizer_reg, loss_cnn, geraet)
    val_ergebnis = bewerte_cnn(reg_lenet, val_loader_bild, loss_cnn, geraet)
    scheduler.step(val_ergebnis["loss"])
    aktuelle_lernrate = optimizer_reg.param_groups[0]["lr"]
    historie_reg.append(
        {
            "Epoche": epoche + 1,
            "train_loss": train_loss,
            "val_loss": val_ergebnis["loss"],
            "lernrate": aktuelle_lernrate,
        }
    )

    if val_ergebnis["loss"] < bester_val_reg - 1e-5:
        bester_val_reg = val_ergebnis["loss"]
        bester_zustand_reg = copy.deepcopy(reg_lenet.state_dict())
        ohne_verbesserung = 0
    else:
        ohne_verbesserung += 1
    if ohne_verbesserung >= 4:
        print("Early Stopping nach Epoche", epoche + 1)
        break

reg_lenet.load_state_dict(bester_zustand_reg)
test_reg_lenet = bewerte_cnn(reg_lenet, test_loader_bild, loss_cnn, geraet)

regularisierungsvergleich = pd.DataFrame(
    [
        {
            "Modell": "LeNet",
            "Bester_Val_Verlust": bester_val,
            "Test_Macro_F1": test_lenet_torch["macro_f1"],
            "Trainierbare_Parameter": sum(p.numel() for p in lenet_torch.parameters() if p.requires_grad),
        },
        {
            "Modell": "LeNet + BN + Dropout",
            "Bester_Val_Verlust": bester_val_reg,
            "Test_Macro_F1": test_reg_lenet["macro_f1"],
            "Trainierbare_Parameter": sum(p.numel() for p in reg_lenet.parameters() if p.requires_grad),
        },
    ]
)
display(regularisierungsvergleich.round(4))
display(pd.DataFrame(historie_reg).tail().round(5))

> **Musterantwort und Interpretation**
>
> Im Trainingsmodus verwendet BatchNorm aktuelle Mini-Batch-Statistiken und aktualisiert gleitende Mittelwerte. Im Evaluationsmodus verwendet es die während des Trainings gesammelten stabilen Statistiken. Bleibt das Modell in train(), hängen Vorhersagen von der Batchzusammensetzung ab und die gespeicherten Statistiken können unbeabsichtigt verändert werden.

### Aufgabe 5: MobileNetV3 Small einfrieren und einen Kopf trainieren

Skalieren Sie kleine Teilmengen der Digits-Bilder auf 64-mal-64, wiederholen Sie den Graustufenkanal dreimal und normalisieren Sie mit den ImageNet-Mittelwerten und -Standardabweichungen.

Versuchen Sie `MobileNet_V3_Small_Weights.DEFAULT` zu laden. Bei fehlendem Internet muss der Code auf zufällige Gewichte zurückfallen und dies dokumentieren. Frieren Sie alle Basisparameter ein, ersetzen Sie die letzte Klassifikationsschicht durch zehn Ausgaben und trainieren Sie nur den neuen Kopf für höchstens zwei Epochen.

In [ ]:
VERSUCHE_VORTRAINIERTE_GEWICHTE = True

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def mobilenet_tensoren(bilder, maximale_anzahl):
    tensor = torch.tensor(bilder[:maximale_anzahl], dtype=torch.float32).unsqueeze(1)
    tensor = F.interpolate(tensor, size=(64, 64), mode="bilinear", align_corners=False)
    tensor = tensor.repeat(1, 3, 1, 1)
    mittel = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
    return (tensor - mittel) / std

transfer_train_x = mobilenet_tensoren(bilder_train_np, 700)
transfer_val_x = mobilenet_tensoren(bilder_val_np, 180)
transfer_test_x = mobilenet_tensoren(bilder_test_np, 250)
transfer_train_y = torch.tensor(labels_train_np[: len(transfer_train_x)], dtype=torch.long)
transfer_val_y = torch.tensor(labels_val_np[: len(transfer_val_x)], dtype=torch.long)
transfer_test_y = torch.tensor(labels_test_np[: len(transfer_test_x)], dtype=torch.long)

transfer_train_loader = DataLoader(
    TensorDataset(transfer_train_x, transfer_train_y),
    batch_size=32,
    shuffle=True,
    generator=torch.Generator().manual_seed(RANDOM_SEED),
)
transfer_val_loader = DataLoader(TensorDataset(transfer_val_x, transfer_val_y), batch_size=64, shuffle=False)
transfer_test_loader = DataLoader(TensorDataset(transfer_test_x, transfer_test_y), batch_size=64, shuffle=False)

verwendet_vortraining = False
try:
    weights = models.MobileNet_V3_Small_Weights.DEFAULT if VERSUCHE_VORTRAINIERTE_GEWICHTE else None
    transfer_torch = models.mobilenet_v3_small(weights=weights)
    verwendet_vortraining = weights is not None
except Exception as fehler:
    print("Vortrainierte Gewichte nicht verfügbar:", type(fehler).__name__)
    print("Offline-Fallback mit zufälligen Gewichten.")
    transfer_torch = models.mobilenet_v3_small(weights=None)

for parameter in transfer_torch.parameters():
    parameter.requires_grad = False

letzte_eingaben = transfer_torch.classifier[-1].in_features
transfer_torch.classifier[-1] = nn.Linear(letzte_eingaben, 10)
transfer_torch = transfer_torch.to(geraet)

trainierbare_parameter = [p for p in transfer_torch.parameters() if p.requires_grad]
optimizer_transfer = torch.optim.Adam(trainierbare_parameter, lr=0.001)

for epoche in range(2):
    train_loss = trainiere_cnn_epoche(
        transfer_torch,
        transfer_train_loader,
        optimizer_transfer,
        loss_cnn,
        geraet,
    )
    val_transfer = bewerte_cnn(transfer_torch, transfer_val_loader, loss_cnn, geraet)
    print(
        f"Epoche {epoche + 1}: Train-Loss={train_loss:.4f}, "
        f"Val-Accuracy={val_transfer['accuracy']:.3f}"
    )

transfer_test_ergebnis = bewerte_cnn(transfer_torch, transfer_test_loader, loss_cnn, geraet)
print("Vortraining verwendet:", verwendet_vortraining)
print("Transfer-Test-Macro-F1:", round(transfer_test_ergebnis["macro_f1"], 3))
print("Trainierbare Parameter:", sum(p.numel() for p in transfer_torch.parameters() if p.requires_grad))

> **Musterantwort und Interpretation**
>
> Ohne vortrainierte Gewichte sind die eingefrorenen Faltungsmerkmale zufällig. Nur der neue Kopf wird angepasst und kann aus diesen zufälligen Darstellungen meist wenig lernen. Der Fallback beweist technische Ausführbarkeit, aber keine Wirkung von Transfer Learning. Ein Bericht muss diesen Status sichtbar ausweisen.

### Aufgabe 6: Integrationsaufgabe: Genauigkeit, Laufzeit und Modellgröße vergleichen

Trainieren Sie eine Pixel-LogReg-Baseline auf den flach dargestellten Digits. Vergleichen Sie Pixel-LogReg, das beste LeNet und MobileNetV3 anhand von Accuracy, Balanced Accuracy, Macro-F1, Gesamtparametern, trainierbaren Parametern und mittlerer Inferenzzeit pro Beispiel.

Messen Sie die Inferenzzeit nach einem Aufwärmdurchlauf mindestens fünfmal. Dokumentieren Sie bei MobileNetV3, ob echte vortrainierte Gewichte verwendet wurden. Treffen Sie eine begründete Wahl für einen CPU-basierten Offline-Einsatz.

In [ ]:
def messe_inferenzzeit(modell, beispiel_batch, wiederholungen=5):
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def messe_inferenzzeit(modell, beispiel_batch, wiederholungen=5):
    modell.eval()
    beispiel_batch = beispiel_batch.to(geraet)
    with torch.no_grad():
        _ = modell(beispiel_batch)
    zeiten = []
    with torch.no_grad():
        for _ in range(wiederholungen):
            start = time.perf_counter()
            _ = modell(beispiel_batch)
            zeiten.append(time.perf_counter() - start)
    return float(np.mean(zeiten) / len(beispiel_batch))

X_pixel_train = bilder_train_np.reshape(len(bilder_train_np), -1)
X_pixel_test = bilder_test_np.reshape(len(bilder_test_np), -1)
pixel_logreg = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=2500, random_state=RANDOM_SEED),
)
pixel_logreg.fit(X_pixel_train, labels_train_np)

start = time.perf_counter()
for _ in range(5):
    pixel_pred = pixel_logreg.predict(X_pixel_test)
pixel_zeit_pro_beispiel = (time.perf_counter() - start) / (5 * len(X_pixel_test))

bestes_lenet = reg_lenet if test_reg_lenet["macro_f1"] >= test_lenet_torch["macro_f1"] else lenet_torch
bestes_lenet_ergebnis = test_reg_lenet if bestes_lenet is reg_lenet else test_lenet_torch
lenet_batch = torch.tensor(bilder_test_np[:128], dtype=torch.float32).unsqueeze(1)
transfer_batch = transfer_test_x[:64]

pixel_schritt = pixel_logreg.named_steps["logisticregression"]
pixel_parameter = pixel_schritt.coef_.size + pixel_schritt.intercept_.size

modellvergleich = pd.DataFrame(
    [
        {
            "Modell": "Pixel-LogReg",
            "Accuracy": accuracy_score(labels_test_np, pixel_pred),
            "Balanced_Accuracy": balanced_accuracy_score(labels_test_np, pixel_pred),
            "Macro_F1": f1_score(labels_test_np, pixel_pred, average="macro"),
            "Parameter_gesamt": pixel_parameter,
            "Parameter_trainierbar": pixel_parameter,
            "Sekunden_pro_Beispiel": pixel_zeit_pro_beispiel,
            "Vortraining": False,
        },
        {
            "Modell": "Bestes LeNet",
            "Accuracy": bestes_lenet_ergebnis["accuracy"],
            "Balanced_Accuracy": bestes_lenet_ergebnis["balanced_accuracy"],
            "Macro_F1": bestes_lenet_ergebnis["macro_f1"],
            "Parameter_gesamt": sum(p.numel() for p in bestes_lenet.parameters()),
            "Parameter_trainierbar": sum(p.numel() for p in bestes_lenet.parameters() if p.requires_grad),
            "Sekunden_pro_Beispiel": messe_inferenzzeit(bestes_lenet, lenet_batch),
            "Vortraining": False,
        },
        {
            "Modell": "MobileNetV3 Small",
            "Accuracy": transfer_test_ergebnis["accuracy"],
            "Balanced_Accuracy": transfer_test_ergebnis["balanced_accuracy"],
            "Macro_F1": transfer_test_ergebnis["macro_f1"],
            "Parameter_gesamt": sum(p.numel() for p in transfer_torch.parameters()),
            "Parameter_trainierbar": sum(p.numel() for p in transfer_torch.parameters() if p.requires_grad),
            "Sekunden_pro_Beispiel": messe_inferenzzeit(transfer_torch, transfer_batch),
            "Vortraining": verwendet_vortraining,
        },
    ]
)
display(modellvergleich.round(6))

> **Musterantwort und Interpretation**
>
> Die konkrete Wahl folgt den gemessenen Kennzahlen. Für kleine, gleichförmige 8-mal-8-Ziffern sind Pixel-LogReg und LeNet meist deutlich kompakter und schneller als MobileNetV3. LeNet nutzt lokale Bildstruktur, während LogReg besonders einfach zu warten und erklären ist. MobileNetV3 lohnt sich eher, wenn die Eingaben natürlichen Bildern ähneln und echte vortrainierte Gewichte einen messbaren Vorteil liefern. Modellgröße, Laufzeit, Offline-Verfügbarkeit und Fehlermuster gehören gleichberechtigt zur Entscheidung.

## Abschlusskontrolle

Prüfen Sie vor dem Abschluss:

- Lassen sich alle Zellen in sinnvoller Reihenfolge ausführen?
- Sind Formen, Datentypen, Wertebereiche und Zufalls-Startwerte dokumentiert?
- Wurden Trainings-, Validierungs- und Testinformationen sauber getrennt?
- Sind Diagramme und Kennzahlen beschriftet und fachlich interpretiert?
- Können Sie erklären, warum die gewählten Methoden zur Aufgabenstellung passen?